In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

RAW = "C:/Users/felip/OneDrive/Documentos/olist/data/raw/"

orders      = pd.read_csv(RAW + "olist_orders_dataset.csv")
customers   = pd.read_csv(RAW + "olist_customers_dataset.csv")
payments    = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")

orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

df = orders.merge(customers, on="customer_id", how="left")
df = df.merge(payments, on="order_id", how="left")
df_entregues = df[df["order_status"] == "delivered"]

print(f"Base pronta: {len(df_entregues):,} pedidos entregues")

Base pronta: 100,757 pedidos entregues


In [2]:
pedidos_por_cliente = df_entregues.groupby("customer_unique_id")["order_id"].nunique()

recompra = pedidos_por_cliente.value_counts().reset_index()
recompra.columns = ["qtd_pedidos", "qtd_clientes"]
recompra["pct"] = recompra["qtd_clientes"] / recompra["qtd_clientes"].sum() * 100

clientes_1_pedido = recompra[recompra["qtd_pedidos"] == 1]["qtd_clientes"].values[0]
clientes_2_mais   = recompra[recompra["qtd_pedidos"] > 1]["qtd_clientes"].sum()

print("=== TAXA DE RECOMPRA ===")
print(f"Clientes com 1 pedido:    {clientes_1_pedido:,} ({clientes_1_pedido/len(pedidos_por_cliente)*100:.1f}%)")
print(f"Clientes com 2+ pedidos:  {clientes_2_mais:,} ({clientes_2_mais/len(pedidos_por_cliente)*100:.1f}%)")
print(f"Máximo de pedidos por cliente: {pedidos_por_cliente.max()}")

fig = px.bar(
    recompra[recompra["qtd_pedidos"] <= 5],
    x="qtd_pedidos", y="qtd_clientes",
    title="Distribuição de Pedidos por Cliente",
    labels={"qtd_pedidos": "Nº de Pedidos", "qtd_clientes": "Nº de Clientes"},
    color="qtd_clientes",
    color_continuous_scale="Blues",
    height=400,
)
fig.show()

=== TAXA DE RECOMPRA ===
Clientes com 1 pedido:    90,557 (97.0%)
Clientes com 2+ pedidos:  2,801 (3.0%)
Máximo de pedidos por cliente: 15


In [3]:
receita_estado = df_entregues.groupby("customer_state")["payment_value"].sum().reset_index()
receita_estado.columns = ["estado", "receita"]
receita_estado = receita_estado.sort_values("receita", ascending=False)

fig2 = px.bar(
    receita_estado,
    x="estado", y="receita",
    title="Receita por Estado",
    labels={"estado": "Estado", "receita": "Receita (R$)"},
    color="receita",
    color_continuous_scale="Blues",
    height=450,
)
fig2.show()

print("\nTop 5 estados por receita:")
print(receita_estado.head(5).to_string(index=False))


Top 5 estados por receita:
estado    receita
    SP 5770266.19
    RJ 2055690.45
    MG 1819277.61
    RS  861802.40
    PR  781919.55


In [4]:
# RFM — Recência, Frequência, Monetário
data_ref = df_entregues["order_purchase_timestamp"].max()

rfm = df_entregues.groupby("customer_unique_id").agg(
    recencia  = ("order_purchase_timestamp", lambda x: (data_ref - x.max()).days),
    frequencia= ("order_id", "nunique"),
    monetario = ("payment_value", "sum")
).reset_index()

# Score RFM (1-5)
rfm["r_score"] = pd.qcut(rfm["recencia"],  q=5, labels=[5,4,3,2,1])
rfm["f_score"] = pd.qcut(rfm["frequencia"].rank(method="first"), q=5, labels=[1,2,3,4,5])
rfm["m_score"] = pd.qcut(rfm["monetario"], q=5, labels=[1,2,3,4,5])
rfm["rfm_score"] = rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)

# Segmentação
def segmentar(row):
    r = int(row["r_score"])
    f = int(row["f_score"])
    m = int(row["m_score"])
    if r >= 4 and f >= 4:
        return "Campeões"
    elif r >= 3 and f >= 3:
        return "Clientes Fiéis"
    elif r >= 4 and f <= 2:
        return "Novos Clientes"
    elif r <= 2 and f >= 3:
        return "Em Risco"
    elif r <= 2 and f <= 2:
        return "Perdidos"
    else:
        return "Potencial"

rfm["segmento"] = rfm.apply(segmentar, axis=1)

seg_count = rfm["segmento"].value_counts().reset_index()
seg_count.columns = ["segmento", "clientes"]

fig3 = px.pie(
    seg_count,
    names="segmento", values="clientes",
    title="Segmentação RFM de Clientes",
    color_discrete_sequence=px.colors.sequential.Blues_r,
    height=500,
)
fig3.show()

print("\nSegmentação RFM:")
print(seg_count.to_string(index=False))


Segmentação RFM:
      segmento  clientes
      Em Risco     22230
Clientes Fiéis     18824
      Perdidos     14986
Novos Clientes     14984
      Campeões     14961
     Potencial      7373
